In [0]:
dbutils.widgets.text("source_system","")
dbutils.widgets.text("table_name","")
dbutils.widgets.text("landing_timestamp","")
dbutils.widgets.text("source_file_format","")

source_system=dbutils.widgets.get("source_system")
table_name= dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")


##print(source_system)
##print(table_name)
###print(landing_timestamp)

In [0]:
landing_path = f"/Volumes/adb_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

# Check whether the landing path (and its parent directories) exist
try:
    dbutils.fs.ls(landing_path)
    path_exists = True
except Exception:
    path_exists = False

if not path_exists:
    print(f"No new files are received. Path does not exist: {landing_path}")
    dbutils.notebook.exit(f"No new files are received for {source_system}/{table_name}/{landing_timestamp}")

In [0]:
if source_file_format =="csv":
   landing_df = spark.read.format("csv").option("header",True).option("inferSchema",True).load(landing_path)
else:
  landing_df = spark.read.format("parquet").load(landing_path)

In [0]:
from pyspark.sql.functions import current_timestamp,lit

updated_df = landing_df.withColumn("landing_timestamp",lit(landing_timestamp))\
                       .withColumn("insert_timestamp",current_timestamp())



In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_caresync.bronze.{table_name}")